In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import os

os.chdir(r"c:\Users\Huy\Documents\UIT\Sem_4\DS102")

In [2]:
df = pd.read_csv('data/clean/87_feature.csv')
df['DATETIME'] = pd.to_datetime(df[['YEAR', 'MONTH', 'DAY', 'HOUR']])

df_0015 = df[(df['YEAR'] >= 2000) & (df['YEAR'] <= 2015)].copy()
df_1624 = df[(df['YEAR'] >= 2016) & (df['YEAR'] <= 2024)].copy()

cols = ['LAT', 'LON']
lat_lon_0015 = df_0015[cols].reset_index().drop(columns='index')
lat_lon_1624 = df_1624[cols].reset_index().drop(columns='index')

In [ ]:
grib_path = 'data/raw/EU(16-24)/data.grib'
ds_1624 = xr.open_dataset(grib_path, engine="cfgrib")
ds_1624

In [ ]:
grib_path = "data/raw/EU(00-15)/data.grib"
ds_0015 = xr.open_dataset(grib_path, engine="cfgrib")
ds_0015

In [ ]:
query_time = xr.DataArray(df_1624['DATETIME'].values, dims="points")
query_lat = xr.DataArray(df_1624['LAT'].values, dims="points")
query_lon = xr.DataArray(df_1624['LON'].values, dims="points")


ds_extract = ds_1624.sel(
    time=query_time, 
    latitude=query_lat, 
    longitude=query_lon, 
    method="nearest"
)

df_grib_features = ds_extract.to_dataframe().reset_index()

df_grib_features['YEAR'] = df_grib_features['time'].dt.year
df_grib_features['MONTH'] = df_grib_features['time'].dt.month
df_grib_features['DAY'] = df_grib_features['time'].dt.day
df_grib_features['HOUR'] = df_grib_features['time'].dt.hour

var_names = list(ds_1624.data_vars)
cols = ['YEAR', 'MONTH', 'DAY', 'HOUR'] + var_names

df_final_1624 = df_grib_features[cols]

print(f"Đã trích xuất thành công {len(df_final_1624)} dòng dữ liệu.")
display(df_final_1624.head())

In [ ]:
query_time = xr.DataArray(df_0015['DATETIME'].values, dims="points")
query_lat = xr.DataArray(df_0015['LAT'].values, dims="points")
query_lon = xr.DataArray(df_0015['LON'].values, dims="points")

ds_extracted = ds_0015.sel(
    time=query_time, 
    latitude=query_lat, 
    longitude=query_lon, 
    method="nearest"
)

df_grib_features = ds_extracted.to_dataframe().reset_index()

df_grib_features['YEAR'] = df_grib_features['time'].dt.year
df_grib_features['MONTH'] = df_grib_features['time'].dt.month
df_grib_features['DAY'] = df_grib_features['time'].dt.day
df_grib_features['HOUR'] = df_grib_features['time'].dt.hour

var_names = list(ds_0015.data_vars)
cols = ['YEAR', 'MONTH', 'DAY', 'HOUR'] + var_names

df_final_0015 = df_grib_features[cols]

print(f"Đã trích xuất thành công {len(df_final_0015)} dòng dữ liệu.")
display(df_final_0015.head())

In [ ]:
data_1624 = pd.concat([lat_lon_1624, df_final_1624], axis=1)
data_1624

In [ ]:
data_0015 = pd.concat([lat_lon_0015, df_final_0015], axis=1)
data_0015

In [ ]:
data = pd.concat([data_0015, data_1624], axis=0)

In [ ]:
data['msl'] = data['msl'] / 100
data['sst'] = data['sst'].interpolate(method='linear')
data['sst'] = data['sst'] - 273.15

In [ ]:
data.to_csv('data/clean/climate_feature.csv', index=False)